# Учись Машина Учись / Learn Machine Learn

<img src="data/lml.png" width=200>

Онлайн-лекции Ильи С. Елисеева: применение методов машинного обучения в анализе данных.

- Канал в Telegram: https://t.me/learn_machine_learn
- YouTube: https://www.youtube.com/channel/UCCwDwKatNitBCVAJajremMQ
- VK: https://vk.com/learn_machine_learn
- GitHub: https://github.com/easyise/learn_machine_learn

---



# Асинхронное программирование в Python

Асинхронное программированое - это основа для решения следующих задач:
- выстраивание очередей задач для внешних сервисов (например, для API);
- распараллеливание задач, которые не требуют блокировки (например, запросы к базе данных);
- обработка большого количества соединений (например, веб-серверы, чат-сервисы, телеграм-боты).

В Python асинхронное программирование реализовано через ключевые слова `async` и `await`. Основные библиотеки для работы с асинхронным кодом:
- `asyncio` - стандартная библиотека для асинхронного программирования в Python;
- `aiohttp` - библиотека для работы с HTTP-запросами в асинхронном режиме;
- `aiomysql` - библиотека для работы с MySQL в асинхронном режиме;
- `aioredis` - библиотека для работы с Redis в асинхронном режиме.

Сегодня я расскажу вам об асинхронном программировании вообще и покажу, как это работает на примере модели-маршрутизатора: она будет решать на сколько сложная задача и отправлять ее на выполнение либо себе самой, либо внешнему провайдеру через OpenAI API.

## 1. Асинхронные функции и корутины

In [ ]:
# обычный последовательный запуск функций
import time

def worker(name):
    print(f"{name}: начало")
    time.sleep(1)
    print(f"{name}: конец")
    print()


worker("A")
worker("B")
worker("C")

### 1.1 Корутины

In [ ]:
# корутина - функция для асинхронного выполнения
async def worker():
    print("Старт")

    time.sleep(1)

    print("Конец")


await worker()

In [ ]:
worker()

In [ ]:
# возвращаемые значения
async def worker_sum(a, b):
    print(f"Сумма {a} + {b} = {a + b}")
    return a + b

x = await worker_sum(2, 2)
x

In [ ]:
# как работает yield в корутинах
async def worker_yield():
    yield 2
    time.sleep(1)
    yield 12
    time.sleep(1)
    yield 85
    time.sleep(1)
    yield 0
    time.sleep(1)
    yield 6

async for value in worker_yield():
    print(value)


### 1.2 Event loop

Event loop - это цикл событий, который управляет выполнением асинхронных задач. Он позволяет запускать несколько задач одновременно, не блокируя выполнение программы.

В Jupyter Notebook event loop уже запущен, поэтому мы можем использовать `await` для вызова асинхронных функций. В обычном Python скрипте нужно использовать `asyncio.run()` для запуска event loop (см. пример с ```04-async-event_loop.py```)

In [ ]:
# две корутины:

# первая
async def worker(name):
    print(f"{name}: старт")

    time.sleep(1)

    print(f"{name}: конец")
    print()

# вторая
async def main():
    await worker("A")
    await worker("B")
    await worker("C")


await main()

In [ ]:
# универсальная функция для запуска корутин в разных средах (Jupyter Notebook, консолька, web, etc...)
import asyncio


def run_worker():
    try:
        loop = asyncio.get_running_loop()
        loop.create_task(worker())  # notebook/active loop
    except RuntimeError:
        asyncio.run(worker())   # normal .py script
    
    
run_worker()

### 1.3 Параллельное выполнение корутин

Используется `asyncio.gather()` для параллельного выполнения нескольких корутин. Это позволяет запускать несколько задач одновременно и ожидать их завершения.

In [ ]:
async def worker(name):
    print(f"{name}: старт")

    await asyncio.sleep(2)

    print(f"{name}: конец")



await asyncio.gather(
    worker("A"),
    worker("B"),
    worker("C")
)



In [ ]:
async def worker(name):
    for i in range(5):
        print(f"{name}: шаг {i}")

        await asyncio.sleep(0.5) # <-- как бы говорит: "дальше там без меня на полсекунды"


await asyncio.gather(
        worker("A"),
        worker("B")
    );

In [ ]:
async def bad_worker(name):
    for i in range(5):
        print(f"{name}: шаг {i}")

        time.sleep(0.5) # <-- блокирует event loop: "всем ждать, пока я посплю полсекунды!"


await asyncio.gather(
        bad_worker("A"),
        bad_worker("B")
    );

### 1.4 Сбор возвращаемых значений



In [ ]:
async def worker_sum(a, b):
    print(f"Сумма {a} + {b} = {a + b}")
    return a + b

x = await asyncio.gather(
    worker_sum(2, 2),
    worker_sum(3, 3),
    worker_sum(4, 4)
)

print(x)  # [4, 6, 8]

### 1.5 Асинхронно != многопоточно

In [ ]:
# не блокирует event loop
async def complex_task():
    await asyncio.sleep(5)
    return "Готово"

# блокирует event loop
async def bad_complex_task():
    time.sleep(5)
    return "Готово"


async def print_progress(task):
    while not task.done():
        print(".", end="", flush=True)
        await asyncio.sleep(0.5)


async def main():
    print("Начало работы...")
    # task = asyncio.create_task(complex_task())
    task = asyncio.create_task(bad_complex_task())
    progress = asyncio.create_task(print_progress(task))

    result = await task
    await progress

    print()
    print(result)


await main()

In [ ]:
async def cpu_heavy(name):
    print(name, "старт")

    s = 0

    for i in range(100_000_000):
        s += i

    print(name, "конец")


async def main():
    await asyncio.gather(
        cpu_heavy("A"),
        cpu_heavy("B")
    )


start = time.perf_counter()

await main()

elapsed = time.perf_counter() - start

print()
print(f"Время: {elapsed:.2f} сек")

In [ ]:
# добавим в цикл await
async def cpu_heavy(name):
    for i in range(10):

        print(name, i)

        for _ in range(10_000_000):
            pass

        await asyncio.sleep(0)


await asyncio.gather(
        cpu_heavy("A"),
        cpu_heavy("B")
    );

**ВЫВОД**

Для вычислительных задач, которые требуют много ресурсов процессора, асинхронное программирование не даст прироста производительности. В таких случаях лучше использовать многопроцессность (`multiprocessing`, `ProcessPoolExecutor`).

Для задач, которые требуют ожидания (например, запросы к внешним сервисам), асинхронное программирование позволяет эффективно использовать ресурсы и ускорять выполнение программы.

## 2. Примитивы

### 2.1 Очереди

Асинхронная очередь, помимо реализации базового функционала очереди FIFO, позволяет ограничивать количество одновременно выполняемых задач.

Помимо этого, также существует ```PriorityQueue``` - очередь с приоритетами, которая позволяет обрабатывать задачи в порядке их приоритета.

In [ ]:
import asyncio
import random


async def producer(queue: asyncio.Queue, count: int):
    for task_id in range(1, count + 1):
        task = {
            "id": task_id,
            "duration": random.uniform(0.5, 2.0),
        }

        await queue.put(task)
        print(f"Добавлена задача {task_id}")

    print("Все задачи добавлены")


async def worker(name: str, queue: asyncio.Queue):
    while True:
        task = await queue.get()

        try:
            print(
                f"{name} начал задачу {task['id']} "
                f"({task['duration']:.1f} сек)"
            )

            await asyncio.sleep(task["duration"])

            print(f"{name} завершил задачу {task['id']}")

        finally:
            queue.task_done()


async def main():
    queue = asyncio.Queue(maxsize=3) # создаем очередь с ограничением на 3 задачи одновременно

    workers = [
        asyncio.create_task(worker(f"Worker-{i}", queue))
        for i in range(1, 4)
    ]

    await producer(queue, count=10) # здесь мы запускаем корутину, которая добавляет задачи в очередь

    await queue.join() # здесь мы ждем, пока все задачи в очереди будут обработаны

    for task in workers: # здесь мы отменяем задачи воркеров, чтобы они завершились после обработки всех задач в очереди
        task.cancel()

    await asyncio.gather(
        *workers,
        return_exceptions=True,
    )

    print("Очередь обработана")


await main()

### 2.2 Семафоры

Семафоры позволяют ограничивать количество одновременно выполняемых задач. Например, если у нас есть ограничение на количество одновременных запросов к внешнему сервису, мы можем использовать семафор для контроля этого количества.

In [ ]:
import asyncio
import random
import time


semaphore = asyncio.Semaphore(3)


async def download_file(file_id: int) -> str:
    print(f"Файл {file_id}: ожидает свободный слот")

    async with semaphore:
        started = time.perf_counter()

        print(f"Файл {file_id}: загрузка началась")

        await asyncio.sleep(
            random.uniform(1, 3)
        )

        elapsed = time.perf_counter() - started

        print(
            f"Файл {file_id}: загружен "
            f"за {elapsed:.1f} сек"
        )

        return f"file_{file_id}.dat"


async def main():
    tasks = [
        download_file(file_id)
        for file_id in range(1, 10+1)
    ]

    results = await asyncio.gather(*tasks)

    print()
    print("Результаты:", results)


await main()

### 2.3 TaskGroup

`TaskGroup` - это новый примитив в Python 3.11, который позволяет группировать задачи и управлять их выполнением. В отличие от `asyncio.gather()`, он позволяет отменять все задачи в группе, если одна из них завершилась с ошибкой.

In [ ]:
async def worker(name: str, delay: float):
    print(f"{name}: старт")

    await asyncio.sleep(delay)

    if name == "B":
        raise RuntimeError(
            "Что-то пошло не так"
        )

    print(f"{name}: завершен")


async def main():
    try:
        async with asyncio.TaskGroup() as tg:
            tg.create_task(
                worker("A", 5)
            )

            tg.create_task(
                worker("B", 2)
            )

            tg.create_task(
                worker("C", 10)
            )

    except* Exception as e:
        print()
        print(
            "TaskGroup отменил "
            "оставшиеся задачи"
        )
        print(e)


await main()

### 2.4 Event

`Event` - это примитив синхронизации, который позволяет одной корутине сигнализировать другой о том, что событие произошло. Это полезно для координации между различными частями программы.m

In [ ]:
import asyncio


shop_open = asyncio.Event()


async def customer(name):
    print(f"{name}: жду открытия магазина")

    await shop_open.wait()

    print(f"{name}: захожу в магазин")


async def open_shop():
    await asyncio.sleep(3)

    print("Магазин открылся!")

    shop_open.set()


await asyncio.gather(
    customer("Иван"),
    customer("Петр"),
    customer("Анна"),
    open_shop(),
)

## 3. Примеры


### 3.1 Одновременная загрузка из Интернета

Пример: загрузка курсов валют за разные годы с сайта `frankfurter.app`. Мы будем использовать асинхронные запросы для ускорения процесса загрузки данных.

Для этого нельзя использовать `requests`, так как она блокирует выполнение программы. Вместо этого мы будем использовать `aiohttp`, которая позволяет делать асинхронные HTTP-запросы.

In [ ]:
%pip install aiohttp

In [ ]:
import asyncio
import aiohttp
import pandas as pd
import matplotlib.pyplot as plt

In [ ]:
YEARS = [2022, 2023, 2024]

BASE = "EUR"
QUOTE = "USD"

API_URL = "https://api.frankfurter.dev/v2/rates"

In [ ]:
async def load_year(
    session: aiohttp.ClientSession,
    year: int,
) -> pd.DataFrame:
    
    params = {
        "from": f"{year}-01-01",
        "to": f"{year}-12-31",
        "base": BASE,
        "quotes": QUOTE,
    }

    print(f"{year}: загрузка началась")

    async with session.get(
        API_URL,
        params=params,
    ) as response:
        response.raise_for_status()
        data = await response.json()

    frame = pd.DataFrame(data)

    frame["date"] = pd.to_datetime(frame["date"])
    frame["day_of_year"] = frame["date"].dt.dayofyear
    frame["year"] = year

    print(f"{year}: загружено {len(frame)} значений")

    return frame[
        ["date", "day_of_year", "year", "rate"]
    ]

In [ ]:
async def load_all_years(
    years: list[int],
) -> list[pd.DataFrame]:
    timeout = aiohttp.ClientTimeout(total=30)

    async with aiohttp.ClientSession(
        timeout=timeout
    ) as session:
        tasks = [
            load_year(session, year)
            for year in years
        ]

        return await asyncio.gather(*tasks)

In [ ]:
timeout = aiohttp.ClientTimeout(total=30)

async with aiohttp.ClientSession(
    timeout=timeout,
    headers={"Accept-Encoding": "gzip, deflate, identity"},
) as session:
    frames = await asyncio.gather(
        *(load_year(session, year) for year in YEARS)
    )

rates = pd.concat(
    frames,
    ignore_index=True,
)

rates

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))

for year, frame in rates.groupby("year"):
    ax.plot(
        frame["day_of_year"],
        frame["rate"],
        label=str(year),
    )

ax.set_title(
    f"Курс {BASE}/{QUOTE} за разные годы"
)
ax.set_xlabel("День года")
ax.set_ylabel(
    f"{QUOTE} за 1 {BASE}"
)
ax.legend(title="Год")
ax.grid(True, alpha=0.3)

plt.show()

### 3.2 Асинхронный запуск нейросетей



In [ ]:
%pip install -q --upgrade openai

In [ ]:
import os
import asyncio
import time

from openai import AsyncOpenAI

In [ ]:
from __config__ import *

# Локальный llama.cpp на домашнем сервере BrainBox.
LOCAL_BASE_URL = "http://brainbox:8080/v1"
LOCAL_API_KEY = "an_api_key"
LOCAL_MODEL = "gemma-4-12b"

In [ ]:
local_client = AsyncOpenAI(
    base_url=LOCAL_BASE_URL,
    api_key=LOCAL_API_KEY,
    timeout=60.0,
    max_retries=1,
)

# cloud_client = AsyncOpenAI(
#     base_url=CLOUD_BASE_URL,
#     api_key=CLOUD_API_KEY,
#     timeout=120.0,
#     max_retries=1,
# )

In [ ]:
async def query_local_model():
    response = await local_client.chat.completions.create(
        model=LOCAL_MODEL,
        messages=[
            {
                "role": "system",
                "content": (
                    "Отвечай кратко, одним предложением."
                ),
            },
            {
                "role": "user",
                "content": (
                    "Объясни асинхронное программирование "
                    "на примере официанта."
                ),
            },
        ],
        temperature=1.0,
    )

    return response.choices[0].message.content


await query_local_model()

In [ ]:
async def stream_llm(
    client: AsyncOpenAI,
    model: str,
    prompt: str,
) -> str:
    stream = await client.chat.completions.create(
        model=model,
        messages=[
            {
                "role": "system",
                "content": (
                    "Отвечай кратко, одним предложением."
                ),
            },
            {
                "role": "user",
                "content": prompt,
            }
        ],
        temperature=1.0,
        stream=True,
    )

    full_answer = ""
    thinking = ''

    async for chunk in stream:
        reasoning_token = chunk.choices[0].delta.reasoning_content if hasattr(chunk.choices[0].delta, 'reasoning_content') else None
        token = chunk.choices[0].delta.content

        if thinking == '' and reasoning_token:
            print('Thinking: ')
        if reasoning_token:
            thinking += reasoning_token
            print(reasoning_token, end="", flush=True)

        if full_answer == '' and token:
            print('\nAnswer: ')
        if token:
            print(token, end="", flush=True)
            full_answer += token

    print()

    return full_answer

In [ ]:
answer = await stream_llm(
    client=local_client,
    model=LOCAL_MODEL,
    prompt=(
        "Объясни асинхронное программирование "
        "на примере официанта."
    ),
)